# sequence_risk_action_cap_screen_v1

Purpose: after `sequence_risk_reward_short_ppo_screen_v1` showed reward PPO drift, test the same sequence-risk information as a **default-off action constraint** on the fold-local BC baseline. This notebook does not promote a runtime overlay. It uses the official qlib backtest path to ask whether capping target exposure in the sequence-risk state can push the evaluated folds toward `+30pp` excess and `<=15%` MaxDD without another PPO run.

Governance:

- **Canonical baseline**: `v19reg_v22_budget_usage_bull_guard_v1` runtime composer.
- **Direct comparison baselines**: `sequence_risk_reward_short_ppo_screen_v1`, `cap10_sequence_risk_target_cap_bc_screen_v1`, `cap10_sequence_risk_checkpoint_penalty_screen_v1`.
- **Role**: `diagnose`.
- **Changed layer**: `model_selection` / `execution-constraint diagnostic`; shared code only adds a default-off qlib backtest overlay helper.
- **Target windows**: `wf_2022` and `wf_2023_2024`, matching the preceding reward and target-cap screens.
- **Accepted trade-off**: a cap must reduce the failed `wf_2022` MaxDD without breaking the already passing `wf_2023_2024` fold.
- **Failure mode**: cap signal fires but execution absorption prevents material change, or MaxDD remains above 15% / relative return falls below +30pp.

Historical reflection before running:

- Closest success case: the no-sequence-cap BC baseline already clears the +30pp relative target on both evaluated folds, so the remaining issue is mainly `wf_2022` drawdown.
- Closest failure case: reward PPO increased `wf_2022` relative return but expanded sequence-state exposure and MaxDD; target-cap labels also failed robustly.
- Why this is not a duplicate: the target is not another PPO or training label. It tests a constrained action interface in the qlib replay path, with trace evidence for cap fire days and changed targets.

Anti-leakage rule:

- The feature family comes from prior diagnostics, but thresholds are recomputed from each replay fold's train target dates with `fold_local_sequence_risk_thresholds()`.
- The notebook does not use OOS thresholds as labels and does not retain actors.

External basis:

- Qlib workflow separation and replay discipline: https://github.com/microsoft/qlib and https://arxiv.org/abs/2009.11189
- PPO and why we stop broad fine-tuning after drift: https://arxiv.org/abs/1707.06347
- Safe policy improvement / conservative RL caution: https://arxiv.org/abs/1712.06924 and https://arxiv.org/abs/2006.04779
- Backtest overfitting / multiple-testing discipline: https://doi.org/10.21314/JCF.2016.322


In [1]:
from __future__ import annotations

import copy
import json
import math
import os
import random
import sys
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.chdir(ROOT)

from src.soxl_quant.app.path_diagnostics import make_backtest_daily_frame
from src.soxl_quant.app.sequence_risk_objective import (
    DEFAULT_TREND_BREAK_SPECS,
    fold_local_sequence_risk_thresholds,
    score_sequence_risk_path,
)
from src.soxl_quant.app.state_objective_bc import (
    StateObjectiveBCTrainingConfig,
    build_state_objective_bc_dataset_from_frame,
    evaluate_bc_actor_slice_fit,
    fit_state_objective_bc_actor_from_frame,
)
from src.soxl_quant.backtest import backtest_profile
from src.soxl_quant.compat import patch_gym_compat
from src.soxl_quant.config import load_profile
from src.soxl_quant.data import build_dataset, feature_columns_for_packs, init_qlib, segment_frame
from src.soxl_quant.settings import QLIB_DATA_PATH

patch_gym_compat()

EXPERIMENT = 'sequence_risk_action_cap_screen_v1'
CARD = ROOT / 'runtime' / 'model_selection' / 'card' / EXPERIMENT
CARD.mkdir(parents=True, exist_ok=True)
SOURCE_REWARD_CARD = ROOT / 'runtime' / 'reward' / 'card' / 'sequence_risk_reward_short_ppo_screen_v1'
SOURCE_TARGET_CAP_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_target_cap_bc_screen_v1'
SOURCE_SELECTOR_CARD = ROOT / 'runtime' / 'model_selection' / 'card' / 'cap10_sequence_risk_checkpoint_penalty_screen_v1'
TARGETS_PATH = SOURCE_TARGET_CAP_CARD / 'baseline_no_sequence_cap_target_contract.csv'
FRAME_PATH = ROOT / 'data' / 'soxl_profile_balanced_rule_residual_v22.pkl'
PROFILE_NAME = 'sparse_release_bc_ppo_screen_v1'
SEED = 20260526
TARGET_RELATIVE_RETURN = 30.0
TARGET_MAX_DD = 15.0
TRAIN_CONFIG = StateObjectiveBCTrainingConfig(steps=300, lr=0.001, batch_size=128, seed=SEED)

WALK_FORWARD_WINDOWS = {
    'wf_2022': {
        'target_fold': 'wf_2022_train',
        'train': ('2010-03-11', '2020-12-31'),
        'valid': ('2021-01-01', '2021-12-31'),
        'test': ('2022-01-01', '2022-12-31'),
    },
    'wf_2023_2024': {
        'target_fold': 'wf_2023_2024_train',
        'train': ('2010-03-11', '2022-12-31'),
        'valid': ('2023-01-01', '2023-12-31'),
        'test': ('2024-01-01', '2024-12-31'),
    },
}

CAP_RECIPES = [
    {'candidate': 'baseline_no_action_cap', 'target_cap': None, 'min_feature_hits': 0, 'description': 'BC baseline without action cap.'},
    {'candidate': 'seq_cap10_two_of_three', 'target_cap': 0.10, 'min_feature_hits': 2, 'description': 'Cap to 10% when at least two trend-break features fire.'},
    {'candidate': 'seq_cap15_two_of_three', 'target_cap': 0.15, 'min_feature_hits': 2, 'description': 'Cap to 15% when at least two trend-break features fire.'},
    {'candidate': 'seq_cap20_two_of_three', 'target_cap': 0.20, 'min_feature_hits': 2, 'description': 'Cap to 20% when at least two trend-break features fire.'},
    {'candidate': 'seq_cap10_any', 'target_cap': 0.10, 'min_feature_hits': 1, 'description': 'Broader 10% cap when any trend-break feature fires.'},
]


def json_safe(value: Any) -> Any:
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.bool_,)):
        return bool(value)
    if isinstance(value, (np.floating,)):
        value = float(value)
    if isinstance(value, float):
        return value if math.isfinite(value) else None
    if isinstance(value, pd.Timestamp):
        return value.isoformat()
    if isinstance(value, Path):
        return str(value)
    return value


def write_json(path: Path, payload: dict[str, Any]) -> None:
    path.write_text(json.dumps(json_safe(payload), indent=2, sort_keys=True, allow_nan=False) + '\n', encoding='utf-8')


def set_seed(seed: int = SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

print('root', ROOT)
print('card', CARD)


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


root /Users/rziiiii/.openclaw/workspace
card /Users/rziiiii/.openclaw/workspace/runtime/model_selection/card/sequence_risk_action_cap_screen_v1


In [2]:
profile = load_profile(PROFILE_NAME)
reward_selection = json.loads((SOURCE_REWARD_CARD / 'selection.json').read_text(encoding='utf-8'))
target_cap_selection = json.loads((SOURCE_TARGET_CAP_CARD / 'selection.json').read_text(encoding='utf-8'))
selector_selection = json.loads((SOURCE_SELECTOR_CARD / 'selection.json').read_text(encoding='utf-8'))
targets = pd.read_csv(TARGETS_PATH, parse_dates=['datetime'])
frame = pd.read_pickle(FRAME_PATH).sort_index()
for col in ('is_rebound_cluster', 'is_downtrend_state', 'release_gate_state', 'drawdown_control_state'):
    if col in targets.columns:
        targets[col] = targets[col].fillna(False).astype(bool)

coverage_rows = []
for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    train_start, train_end = map(pd.Timestamp, spec['train'])
    valid_start, valid_end = map(pd.Timestamp, spec['valid'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    coverage_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'valid_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(valid_start, valid_end)].shape[0]),
        'test_overlap_rows': int(targets[targets['fold'].eq(spec['target_fold']) & targets['datetime'].between(test_start, test_end)].shape[0]),
        'release_gate_rows': int(fold_targets.get('release_gate_state', pd.Series(False, index=fold_targets.index)).sum()),
        'downtrend_rows': int(fold_targets.get('is_downtrend_state', pd.Series(False, index=fold_targets.index)).sum()),
        'drawdown_control_rows': int(fold_targets.get('drawdown_control_state', pd.Series(False, index=fold_targets.index)).sum()),
        'train_start': spec['train'][0],
        'train_end': spec['train'][1],
        'valid_start': spec['valid'][0],
        'valid_end': spec['valid'][1],
        'test_start': spec['test'][0],
        'test_end': spec['test'][1],
    })
coverage = pd.DataFrame(coverage_rows)
coverage.to_csv(CARD / 'coverage.csv', index=False)
write_json(CARD / 'coverage.json', {
    'experiment': EXPERIMENT,
    'profile': PROFILE_NAME,
    'targets_path': str(TARGETS_PATH.relative_to(ROOT)),
    'feature_frame_path': str(FRAME_PATH.relative_to(ROOT)),
    'source_reward_decision': reward_selection.get('decision'),
    'source_target_cap_decision': target_cap_selection.get('decision'),
    'source_selector_decision': selector_selection.get('decision'),
    'windows': coverage.to_dict('records'),
})
display(coverage)


,fold,target_fold,train_target_rows,valid_overlap_rows,test_overlap_rows,release_gate_rows,downtrend_rows,drawdown_control_rows,train_start,train_end,valid_start,valid_end,test_start,test_end
0,wf_2022,wf_2022_train,253,252,0,40,118,49,2010-03-11,2020-12-31,2021-01-01,2021-12-31,2022-01-01,2022-12-31
1,wf_2023_2024,wf_2023_2024_train,756,0,0,40,478,49,2010-03-11,2022-12-31,2023-01-01,2023-12-31,2024-01-01,2024-12-31


In [3]:
def overlay_config_for_recipe(recipe: dict[str, Any], thresholds: dict[str, dict[str, Any]]) -> dict[str, Any] | None:
    if recipe['target_cap'] is None:
        return None
    return {
        'type': 'sequence_risk_action_cap',
        'target_cap': float(recipe['target_cap']),
        'sequence_risk_min_feature_hits': int(recipe['min_feature_hits']),
        'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
        'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
        'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
    }


def official_metrics(bt: dict[str, Any], price_series: pd.Series) -> dict[str, float | str]:
    effective_start, effective_end = bt.get('effective_test_seg', ('', ''))
    prices = price_series.loc[str(effective_start):str(effective_end)].dropna().astype(float)
    bnh = (float(prices.iloc[-1]) / float(prices.iloc[0]) - 1.0) * 100.0 if len(prices) >= 2 and float(prices.iloc[0]) else float('nan')
    ret = float(bt.get('return', float('nan')))
    pos_hist = np.asarray(bt.get('position_history', []), dtype=float)
    return {
        'return': ret,
        'sharpe': float(bt.get('sharpe', float('nan'))),
        'max_dd': float(bt.get('max_dd', float('nan'))),
        'buy_and_hold_return': bnh,
        'relative_return': ret - bnh if math.isfinite(ret) and math.isfinite(bnh) else float('nan'),
        'avg_pos': float(np.mean(pos_hist)) if pos_hist.size else 0.0,
        'stress_avg_pos': float(np.mean(pos_hist[pos_hist > 0.0])) if pos_hist.size and np.any(pos_hist > 0.0) else 0.0,
        'effective_start': str(effective_start),
        'effective_end': str(effective_end),
    }


def target_gate(metrics: dict[str, Any]) -> bool:
    return bool(float(metrics.get('relative_return', float('-inf'))) >= TARGET_RELATIVE_RETURN and float(metrics.get('max_dd', float('inf'))) <= TARGET_MAX_DD)


def trace_metrics(trace: list[dict[str, Any]]) -> dict[str, float | int]:
    if not trace:
        return {
            'trace_rows': 0,
            'overlay_ready_days': 0,
            'target_cap_changed_days': 0,
            'order_days': 0,
            'blocked_by_trade_lock_days': 0,
            'blocked_by_min_rebalance_days': 0,
        }
    frame_trace = pd.DataFrame(trace)
    changed = pd.to_numeric(frame_trace.get('target_after_overlay', 0.0), errors='coerce') < pd.to_numeric(frame_trace.get('target_before_overlay', 0.0), errors='coerce') - 1e-9
    return {
        'trace_rows': int(len(frame_trace)),
        'overlay_ready_days': int(frame_trace.get('overlay_gate_ready', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
        'target_cap_changed_days': int(changed.fillna(False).sum()),
        'order_days': int(pd.to_numeric(frame_trace.get('order_count', pd.Series(0, index=frame_trace.index)), errors='coerce').fillna(0).gt(0).sum()),
        'blocked_by_trade_lock_days': int(frame_trace.get('blocked_by_trade_lock', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
        'blocked_by_min_rebalance_days': int(frame_trace.get('blocked_by_min_rebalance', pd.Series(False, index=frame_trace.index)).fillna(False).astype(bool).sum()),
    }


def prefixed(prefix: str, metrics: dict[str, Any]) -> dict[str, Any]:
    return {f'{prefix}_{key}': value for key, value in metrics.items()}


In [4]:
init_qlib(QLIB_DATA_PATH)
feature_cols = feature_columns_for_packs(profile.feature.packs, profile.name)
feature_frame_for_daily = frame[['anchor_slope_180', 'anchor_gap_180', 'trend_gap_63']].copy()

fold_rows = []
slice_rows = []
threshold_rows = []
daily_paths: dict[tuple[str, str], pd.DataFrame] = {}
trace_frames = []

for fold_name, spec in WALK_FORWARD_WINDOWS.items():
    print(f"\n[fold:{fold_name}] train={spec['train']} test={spec['test']}")
    set_seed(SEED)
    train_start, train_end = map(pd.Timestamp, spec['train'])
    test_start, test_end = map(pd.Timestamp, spec['test'])
    fold_targets = targets[
        targets['fold'].eq(spec['target_fold'])
        & targets['datetime'].between(train_start, train_end)
    ].copy()
    if fold_targets.empty:
        fold_rows.append({'fold': fold_name, 'candidate': 'all', 'status': 'skipped_no_targets'})
        continue
    thresholds = fold_local_sequence_risk_thresholds(frame, fold_targets['datetime'], specs=DEFAULT_TREND_BREAK_SPECS)
    threshold_rows.append({
        'fold': fold_name,
        'target_fold': spec['target_fold'],
        'train_target_rows': int(len(fold_targets)),
        'thresholds_json': json.dumps(json_safe(thresholds), sort_keys=True),
        'sequence_risk_anchor_slope_max': float(thresholds['anchor_slope_180']['threshold']),
        'sequence_risk_anchor_gap_max': float(thresholds['anchor_gap_180']['threshold']),
        'sequence_risk_trend_gap_min': float(thresholds['trend_gap_63']['threshold']),
    })

    fit = fit_state_objective_bc_actor_from_frame(
        frame,
        targets=fold_targets,
        profile=profile,
        train_config=TRAIN_CONFIG,
        sample_weight_config=None,
        sample_size=0,
    )
    dataset_for_bc = build_state_objective_bc_dataset_from_frame(frame, targets=fold_targets, profile=profile, sample_size=0)
    slice_metrics = evaluate_bc_actor_slice_fit(
        fit.actor,
        fit.obs,
        fit.actions,
        dataset_for_bc.metadata,
        slice_cols=('release_gate_state', 'is_downtrend_state', 'drawdown_control_state', 'is_rebound_cluster'),
    )
    tagged_slices = slice_metrics.copy()
    tagged_slices.insert(0, 'fold', fold_name)
    slice_rows.append(tagged_slices)

    dataset = build_dataset(
        train_seg=spec['train'],
        valid_seg=spec['valid'],
        test_seg=spec['test'],
        feature_packs=tuple(profile.feature.packs),
        profile_name=profile.name,
    )
    test_df = segment_frame(dataset, 'test')
    price_series = dataset.handler.price_frame.droplevel('instrument')['price'].sort_index().loc[spec['test'][0]:spec['test'][1]]

    for recipe in CAP_RECIPES:
        candidate = str(recipe['candidate'])
        overlay = overlay_config_for_recipe(recipe, thresholds)
        bt = backtest_profile(
            fit.actor,
            test_df,
            price_series,
            position_cap_config=profile.position_cap,
            position_budget_config=profile.position_budget,
            execution_config=profile.execution,
            asset_idx=0,
            feature_columns=feature_cols,
            model_config=profile.model,
            mode_router_config=profile.mode_router,
            test_seg=spec['test'],
            overlay_config=overlay,
            trace_decisions=True,
        )
        metrics = official_metrics(bt, price_series)
        daily = make_backtest_daily_frame(bt, label=f'{candidate}_{fold_name}', feature_frame=feature_frame_for_daily)
        daily_paths[(candidate, fold_name)] = daily
        seq_score = score_sequence_risk_path(daily, thresholds=thresholds, mode='two_of_three')
        trace = pd.DataFrame(bt.get('decision_trace', []))
        if not trace.empty:
            trace.insert(0, 'candidate', candidate)
            trace.insert(0, 'fold', fold_name)
            trace_frames.append(trace)
        fold_rows.append({
            'candidate': candidate,
            'fold': fold_name,
            'status': 'evaluated',
            'target_cap': np.nan if recipe['target_cap'] is None else float(recipe['target_cap']),
            'min_feature_hits': int(recipe['min_feature_hits']),
            'description': str(recipe['description']),
            'bc_after_accuracy': float(fit.metrics['after_accuracy']),
            'bc_after_loss': float(fit.metrics['after_loss']),
            'target_gate': target_gate(metrics),
            **metrics,
            **prefixed('trace', trace_metrics(bt.get('decision_trace', []))),
            **prefixed('sequence', seq_score),
        })

thresholds_df = pd.DataFrame(threshold_rows)
metrics_df = pd.DataFrame(fold_rows)
slices_df = pd.concat(slice_rows, ignore_index=True, sort=False) if slice_rows else pd.DataFrame()
traces_df = pd.concat(trace_frames, ignore_index=True, sort=False) if trace_frames else pd.DataFrame()
thresholds_df.to_csv(CARD / 'fold_sequence_thresholds.csv', index=False)
metrics_df.to_csv(CARD / 'action_cap_metrics.csv', index=False)
slices_df.to_csv(CARD / 'bc_slice_metrics.csv', index=False)
traces_df.to_csv(CARD / 'decision_trace.csv', index=False)

display(metrics_df)


[43843:MainThread](2026-05-26 10:44:46,977) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[43843:MainThread](2026-05-26 10:44:47,642) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[43843:MainThread](2026-05-26 10:44:47,642) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}



[fold:wf_2022] train=('2010-03-11', '2020-12-31') test=('2022-01-01', '2022-12-31')


/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


...

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  ret

/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)
/Users/rziiiii/.openclaw/workspace/qlib_venv/lib/python3.11/site-packages/qlib/utils/index_data.py:492: RuntimeWarning: Mean of empty slice
  return np.nanmean(self.data)


,candidate,fold,status,target_cap,min_feature_hits,description,bc_after_accuracy,bc_after_loss,target_gate,return,...,sequence_sequence_position_lift,sequence_sequence_max_dd_pct,sequence_drawdown_breach_rows,sequence_sequence_breach_rows,sequence_sequence_breach_capture,sequence_sequence_hit_count_mean,sequence_sequence_exposure_loss,sequence_sequence_drawdown_loss,sequence_sequence_breach_capture_loss,sequence_sequence_path_penalty
0,baseline_no_action_cap,wf_2022,evaluated,NaN,0,BC baseline without action cap.,0.833992,1.073453,False,-16.534504,...,0.115923,18.396429,30,30,1.0,1.676,0.777586,0.226429,0.777586,0.557123
1,seq_cap10_two_of_three,wf_2022,evaluated,0.10,2,Cap to 10% when at least two trend-break featu...,0.833992,1.073453,True,-7.725714,...,0.036506,11.530076,0,0,0.0,1.676,0.000000,0.000000,0.000000,0.000000
2,seq_cap15_two_of_three,wf_2022,evaluated,0.15,2,Cap to 15% when at least two trend-break featu...,0.833992,1.073453,False,-9.884809,...,0.065006,15.157706,2,2,1.0,1.676,0.259129,0.010514,0.259129,0.159683
3,seq_cap20_two_of_three,wf_2022,evaluated,0.20,2,Cap to 20% when at least two trend-break featu...,0.833992,1.073453,False,-9.500257,...,0.091496,17.330761,18,18,1.0,1.676,0.533319,0.155384,0.533319,0.382145
4,seq_cap10_any,wf_2022,evaluated,0.10,1,Broader 10% cap when any trend-break feature f...,0.833992,1.073453,True,-7.725714,...,0.036506,11.530076,0,0,0.0,1.676,0.000000,0.000000,0.000000,0.000000
5,baseline_no_action_cap,wf_2023_2024,evaluated,NaN,0,BC baseline without action cap.,0.768212,1.146839,False,-5.811867,...,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000
6,seq_cap10_two_of_three,wf_2023_2024,evaluated,0.10,2,Cap to 10% when at least two trend-break featu...,0.768212,1.146839,False,-5.811867,...,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000
7,seq_cap15_two_of_three,wf_2023_2024,evaluated,0.15,2,Cap to 15% when at least two trend-break featu...,0.768212,1.146839,False,-5.811867,...,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000
8,seq_cap20_two_of_three,wf_2023_2024,evaluated,0.20,2,Cap to 20% when at least two trend-break featu...,0.768212,1.146839,False,-5.811867,...,-0.145913,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000
9,seq_cap10_any,wf_2023_2024,evaluated,0.10,1,Broader 10% cap when any trend-break feature f...,0.768212,1.146839,False,-6.322983,...,-0.107134,0.000000,0,0,0.0,0.000,0.000000,0.000000,0.000000,0.000000


In [5]:
def candidate_loss(group: pd.DataFrame) -> float:
    if group.empty:
        return float('inf')
    loss = 0.0
    for row in group.to_dict('records'):
        rel = float(row.get('relative_return', float('-inf')))
        dd = float(row.get('max_dd', float('inf')))
        loss += max(0.0, TARGET_RELATIVE_RETURN - rel) / TARGET_RELATIVE_RETURN
        loss += 2.0 * max(0.0, dd - TARGET_MAX_DD) / TARGET_MAX_DD
        if int(row.get('trace_target_cap_changed_days', 0)) == 0 and row.get('candidate') != 'baseline_no_action_cap':
            loss += 0.25
    complexity = float(group['min_feature_hits'].max()) if 'min_feature_hits' in group else 0.0
    return float(loss + 0.001 * complexity)

summary_rows = []
evaluated = metrics_df[metrics_df['status'].eq('evaluated')].copy() if not metrics_df.empty else pd.DataFrame()
for candidate, group in evaluated.groupby('candidate', sort=False):
    target_pass_count = int(group['target_gate'].astype(bool).sum())
    full_gate = bool(len(group) == len(WALK_FORWARD_WINDOWS) and target_pass_count == len(group))
    summary_rows.append({
        'candidate': candidate,
        'fold_count': int(len(group)),
        'target_pass_count': target_pass_count,
        'full_gate': full_gate,
        'candidate_loss': candidate_loss(group),
        'min_relative_return': float(group['relative_return'].min()),
        'max_max_dd': float(group['max_dd'].max()),
        'mean_avg_pos': float(group['avg_pos'].mean()),
        'total_overlay_ready_days': int(group['trace_overlay_ready_days'].sum()),
        'total_target_cap_changed_days': int(group['trace_target_cap_changed_days'].sum()),
        'max_sequence_path_penalty': float(group['sequence_sequence_path_penalty'].max()),
        'wf_2022_relative_return': float(group[group['fold'].eq('wf_2022')]['relative_return'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2022_max_dd': float(group[group['fold'].eq('wf_2022')]['max_dd'].iloc[0]) if 'wf_2022' in set(group['fold']) else np.nan,
        'wf_2022_target_cap_changed_days': int(group[group['fold'].eq('wf_2022')]['trace_target_cap_changed_days'].iloc[0]) if 'wf_2022' in set(group['fold']) else 0,
        'wf_2023_2024_relative_return': float(group[group['fold'].eq('wf_2023_2024')]['relative_return'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
        'wf_2023_2024_max_dd': float(group[group['fold'].eq('wf_2023_2024')]['max_dd'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else np.nan,
        'wf_2023_2024_target_cap_changed_days': int(group[group['fold'].eq('wf_2023_2024')]['trace_target_cap_changed_days'].iloc[0]) if 'wf_2023_2024' in set(group['fold']) else 0,
    })
summary_df = pd.DataFrame(summary_rows).sort_values(['full_gate', 'candidate_loss'], ascending=[False, True]).reset_index(drop=True)
summary_df.to_csv(CARD / 'candidate_summary.csv', index=False)

baseline = summary_df[summary_df['candidate'].eq('baseline_no_action_cap')].iloc[0].to_dict() if not summary_df[summary_df['candidate'].eq('baseline_no_action_cap')].empty else {}
full_gate_rows = summary_df[summary_df['full_gate'].astype(bool)] if not summary_df.empty else pd.DataFrame()
robust_improvement_rows = pd.DataFrame()
if baseline and not summary_df.empty:
    robust_improvement_rows = summary_df[
        summary_df['candidate'].ne('baseline_no_action_cap')
        & (summary_df['wf_2022_max_dd'] < float(baseline['wf_2022_max_dd']))
        & (summary_df['max_max_dd'] <= TARGET_MAX_DD)
        & (summary_df['wf_2022_target_cap_changed_days'] > 0)
        & (summary_df['wf_2023_2024_max_dd'] <= max(float(baseline['wf_2023_2024_max_dd']) + 2.0, TARGET_MAX_DD))
    ].copy()

if not full_gate_rows.empty:
    selected = full_gate_rows.iloc[0].to_dict()
    decision = 'sequence_risk_action_cap_gate_passed_no_promotion'
    next_mechanism = 'run_full_fixed_split_plus_wf2020_validation_and_actor_retention_review'
elif not robust_improvement_rows.empty:
    selected = robust_improvement_rows.sort_values([
        'wf_2023_2024_target_cap_changed_days',
        'candidate_loss',
    ]).iloc[0].to_dict()
    decision = 'sequence_risk_action_cap_drawdown_candidate_metric_mismatch_no_promotion'
    next_mechanism = 'reconcile_official_relative_vs_existing_gate_then_test_selected_action_cap_on_fixed_split_and_wf2020'
elif baseline:
    selected = baseline
    decision = 'sequence_risk_action_cap_absorbed_or_no_improvement_no_promotion'
    next_mechanism = 'do_not_use_action_cap_as_runtime_overlay; move sequence-risk into checkpoint scoring or stricter actor loss'
else:
    selected = summary_df.iloc[0].to_dict() if not summary_df.empty else {}
    decision = 'sequence_risk_action_cap_screen_no_evaluable_candidate_no_promotion'
    next_mechanism = 'fix_action_cap_screen_inputs_before_more_training'

selected_candidate = str(selected.get('candidate', '')) if selected else ''
for (candidate, fold_name), daily in daily_paths.items():
    if candidate in {'baseline_no_action_cap', selected_candidate}:
        daily.to_csv(CARD / f'{candidate}_{fold_name}_daily_path.csv', index=False)

display(summary_df)


,candidate,fold_count,target_pass_count,full_gate,candidate_loss,min_relative_return,max_max_dd,mean_avg_pos,total_overlay_ready_days,total_target_cap_changed_days,max_sequence_path_penalty,wf_2022_relative_return,wf_2022_max_dd,wf_2022_target_cap_changed_days,wf_2023_2024_relative_return,wf_2023_2024_max_dd,wf_2023_2024_target_cap_changed_days
0,seq_cap10_any,2,1,False,1.199878,-5.966348,11.530076,0.093168,242,145,0.000000,78.862358,11.530076,104,-5.966348,9.733743,41
1,seq_cap10_two_of_three,2,1,False,1.433841,-5.455232,13.472617,0.109765,183,104,0.000000,78.862358,11.530076,104,-5.455232,13.472617,0
2,seq_cap15_two_of_three,2,0,False,1.454868,-5.455232,15.157706,0.122428,183,104,0.159683,76.703263,15.157706,104,-5.455232,13.472617,0
3,baseline_no_action_cap,2,0,False,1.634698,-5.455232,18.396429,0.142013,0,0,0.557123,70.053568,18.396429,0,-5.455232,13.472617,0
4,seq_cap20_two_of_three,2,0,False,1.744609,-5.455232,17.330761,0.130909,183,43,0.382145,77.087814,17.330761,43,-5.455232,13.472617,0


In [6]:
sidecars = [
    'coverage.csv',
    'coverage.json',
    'fold_sequence_thresholds.csv',
    'bc_slice_metrics.csv',
    'action_cap_metrics.csv',
    'candidate_summary.csv',
    'decision_trace.csv',
    'selection.json',
    'summary.json',
]
for name in ['baseline_no_action_cap', selected_candidate]:
    if name:
        for fold_name in WALK_FORWARD_WINDOWS:
            path_name = f'{name}_{fold_name}_daily_path.csv'
            if (CARD / path_name).exists() and path_name not in sidecars:
                sidecars.append(path_name)

selection = {
    'experiment': EXPERIMENT,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'decision': decision,
    'next_mechanism': next_mechanism,
    'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
    'direct_comparison_baseline': [
        'sequence_risk_reward_short_ppo_screen_v1',
        'cap10_sequence_risk_target_cap_bc_screen_v1',
        'cap10_sequence_risk_checkpoint_penalty_screen_v1',
    ],
    'role': 'diagnose',
    'profile': PROFILE_NAME,
    'changed_layer': ['model_selection', 'execution_constraint_diagnostic', 'src_capability'],
    'promotion_allowed': False,
    'training_allowed_from_this_card': False,
    'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic BC actors',
    'promotion_blocker': 'This is a two-fold action-cap screen and uses official qlib relative-return diagnostics that disagree with prior evaluate_split relative gates. Promotion requires metric reconciliation, fixed-split, wf_2020/wf_2022/wf_2023_2024, production execution, and registry review.',
    'target_relative_return_pp': TARGET_RELATIVE_RETURN,
    'target_max_dd_pct': TARGET_MAX_DD,
    'selected_candidate': selected_candidate,
    'selected_full_gate': bool(selected.get('full_gate', False)) if selected else False,
    'selected_candidate_loss': selected.get('candidate_loss') if selected else None,
    'selected_min_relative_return': selected.get('min_relative_return') if selected else None,
    'selected_max_max_dd': selected.get('max_max_dd') if selected else None,
    'selected_wf_2022_relative_return': selected.get('wf_2022_relative_return') if selected else None,
    'selected_wf_2022_max_dd': selected.get('wf_2022_max_dd') if selected else None,
    'selected_wf_2023_2024_relative_return': selected.get('wf_2023_2024_relative_return') if selected else None,
    'selected_wf_2023_2024_max_dd': selected.get('wf_2023_2024_max_dd') if selected else None,
    'selected_wf_2022_target_cap_changed_days': selected.get('wf_2022_target_cap_changed_days') if selected else None,
    'selected_wf_2023_2024_target_cap_changed_days': selected.get('wf_2023_2024_target_cap_changed_days') if selected else None,
    'baseline_wf_2022_max_dd': baseline.get('wf_2022_max_dd') if baseline else None,
    'baseline_wf_2023_2024_max_dd': baseline.get('wf_2023_2024_max_dd') if baseline else None,
    'full_gate_candidate_count': int(len(full_gate_rows)) if 'full_gate_rows' in globals() else 0,
    'robust_improvement_candidate_count': int(len(robust_improvement_rows)) if 'robust_improvement_rows' in globals() else 0,
    'source_reward_decision': reward_selection.get('decision'),
    'source_target_cap_decision': target_cap_selection.get('decision'),
    'source_selector_decision': selector_selection.get('decision'),
    'anti_leakage_note': 'Thresholds are recomputed from each fold train target dates; the overlay is default-off and no OOS thresholds or actors are retained.',
    'external_sources': [
        'https://github.com/microsoft/qlib',
        'https://arxiv.org/abs/2009.11189',
        'https://arxiv.org/abs/1707.06347',
        'https://arxiv.org/abs/1712.06924',
        'https://arxiv.org/abs/2006.04779',
        'https://doi.org/10.21314/JCF.2016.322',
    ],
    'sidecars': sidecars,
}

summary_payload = {
    'experiment': EXPERIMENT,
    'decision': decision,
    'next_mechanism': next_mechanism,
    'coverage': coverage.to_dict('records'),
    'thresholds': thresholds_df.to_dict('records'),
    'candidate_summary': summary_df.to_dict('records'),
    'metrics': metrics_df.to_dict('records'),
    'selected': selected,
    'baseline': baseline,
    'selection': selection,
    'risk_notes': [
        'No runtime overlay or actor is promoted from this notebook.',
        'The screen uses official qlib replay metrics plus decision trace evidence for cap fire and target-change days.',
        'A two-fold drawdown-control candidate is not enough for production; metric-source reconciliation and wider fixed-split/wf_2020 validation are required.',
    ],
    'conclusion': [],
}
if evaluated.empty:
    summary_payload['conclusion'].append('No candidates were evaluated; no promotion or training continuation is allowed.')
else:
    for row in summary_df.to_dict('records'):
        summary_payload['conclusion'].append(
            f"{row['candidate']}: min rel {row.get('min_relative_return', float('nan')):.2f}pp, "
            f"max MaxDD {row.get('max_max_dd', float('nan')):.2f}%, "
            f"target changes {int(row.get('total_target_cap_changed_days', 0))}, full_gate {bool(row.get('full_gate', False))}."
        )
write_json(CARD / 'selection.json', selection)
write_json(CARD / 'summary.json', summary_payload)

lines = ['## Conclusion', '', f"Decision: `{selection['decision']}`.", '']
if selected:
    lines.append(
        f"Selected candidate: `{selected_candidate}` with full gate `{bool(selected.get('full_gate', False))}`, "
        f"min relative `{selected.get('min_relative_return'):.2f}pp`, max MaxDD `{selected.get('max_max_dd'):.2f}%`, "
        f"target-change days `{int(selected.get('total_target_cap_changed_days', 0))}`; "
    f"wf_2023_2024 target-change days `{int(selected.get('wf_2023_2024_target_cap_changed_days', 0))}`."
    )
    lines.append('')
    lines.append(
        f"wf_2022: rel `{selected.get('wf_2022_relative_return'):.2f}pp`, MaxDD `{selected.get('wf_2022_max_dd'):.2f}%`; "
        f"wf_2023_2024: rel `{selected.get('wf_2023_2024_relative_return'):.2f}pp`, MaxDD `{selected.get('wf_2023_2024_max_dd'):.2f}%`."
    )
else:
    lines.append('No candidate was evaluated.')
lines.extend([
    '',
    'Risk note: no runtime promotion and no retained actor. This is an action-constraint diagnostic only; metric-source reconciliation and wider validation are required even if drawdown improves.',
    '',
    f"Next action: `{selection['next_mechanism']}`.",
    '',
    'Artifacts: `action_cap_metrics.csv`, `candidate_summary.csv`, `decision_trace.csv`, fold daily paths, `selection.json`, `summary.json`.',
])
display(Markdown(chr(10).join(lines)))
selection


## Conclusion

Decision: `sequence_risk_action_cap_drawdown_candidate_metric_mismatch_no_promotion`.

Selected candidate: `seq_cap10_two_of_three` with full gate `False`, min relative `-5.46pp`, max MaxDD `13.47%`, target-change days `104`; wf_2023_2024 target-change days `0`.

wf_2022: rel `78.86pp`, MaxDD `11.53%`; wf_2023_2024: rel `-5.46pp`, MaxDD `13.47%`.

Risk note: no runtime promotion and no retained actor. This is an action-constraint diagnostic only; metric-source reconciliation and wider validation are required even if drawdown improves.

Next action: `reconcile_official_relative_vs_existing_gate_then_test_selected_action_cap_on_fixed_split_and_wf2020`.

Artifacts: `action_cap_metrics.csv`, `candidate_summary.csv`, `decision_trace.csv`, fold daily paths, `selection.json`, `summary.json`.

{'experiment': 'sequence_risk_action_cap_screen_v1',
 'created_at': '2026-05-26T02:45:49.070246+00:00',
 'decision': 'sequence_risk_action_cap_drawdown_candidate_metric_mismatch_no_promotion',
 'next_mechanism': 'reconcile_official_relative_vs_existing_gate_then_test_selected_action_cap_on_fixed_split_and_wf2020',
 'canonical_baseline': 'v19reg_v22_budget_usage_bull_guard_v1',
 'direct_comparison_baseline': ['sequence_risk_reward_short_ppo_screen_v1',
  'cap10_sequence_risk_target_cap_bc_screen_v1',
  'cap10_sequence_risk_checkpoint_penalty_screen_v1'],
 'role': 'diagnose',
 'profile': 'sparse_release_bc_ppo_screen_v1',
 'changed_layer': ['model_selection',
  'execution_constraint_diagnostic',
  'src_capability'],
 'promotion_allowed': False,
 'training_allowed_from_this_card': False,
 'actor_retention': 'no actor bundle is promoted or retained by this notebook; rerun the notebook to regenerate diagnostic BC actors',
 'promotion_blocker': 'This is a two-fold action-cap screen and uses 